In [6]:
import numpy as np 
import pandas as pd 
from tqcenter import tq
import time
import json
import os
from IPython import get_ipython

"""
    参数设置
"""
codes = ["688318.SH"] #传入的股票代码格式必须是标准格式：6位数+市场后缀（.SH/.SZ/.JJ等）
startime = "20250620" #传入的时间格式必须是：YYYYMMDD 或 YYYYMMDDHHMMSS 
endtime = "20250801"
period = '1d' #K线周期：1d/1w/1m/5m/15m/30m/60m等
dividend_type='none' #复权类型：none-不复权，front-前复权，back-后复权
symbol_a = "000001.SZ"
symbol_hk = "01658.HK"


def get_notebook_path():
    try:
        # Works in Jupyter Notebook
        ipython = get_ipython()
        if ipython is None:
            return os.getcwd()
        
        # Try to get the notebook path
        try:
            # Method 1: Using IPython magic
            notebook_path = ipython.magic("%pwd")
            return notebook_path
        except:
            pass
        
        # Method 2: Fallback to current working directory
        return os.getcwd()
    except:
        return os.getcwd()

# Set __file__ to the notebook path
__file__ = get_notebook_path()

In [3]:
#初始化
tq.initialize(__file__) #所有策略连接通达信客户端都必须调用此函数进行初始化

'''
    刷新行情缓存 刷新后5分钟内取最新report和k线数据不会触发刷新
    force参数表示是否强制刷新 默认False
    market参数表示指定市场刷新
    'AG'表示A股 'HK'表示港股 'US'表示美股 'QH'表示期货  'QQ'表示期权  'ZZ'表示中证国证指数
'''
refresh_cache = tq.refresh_cache()
print(refresh_cache)


TQ数据接口初始化成功，使用路径: c:\Users\roger\src\openbb\tdx\docs
{"ErrorId":"0","Msg":"Refresh Cache Success.","run_id":"3"}



## 1. 股票相关模型 (Equity Models)
### 1.1 Equity Historical Price (股票历史价格)

In [8]:
'''
    获取K线数据  获取K线数据需要先在客户端中下载对应盘后数据，调用会触发客户端刷新数据，耗时过长请耐心等待
    field_list可以筛选返回字段，默认返回全部字段 比如 field_list=['Open','Close'] 就是只取开盘价和收盘价
    count可以设置每只股票取的数据量
    暂时不支持获取分笔数据
    开高低收单位为元，成交量单位为手，成交额单位为万元
'''
df = tq.get_market_data(
        field_list=[],
        stock_list=['688318.SH'],
        start_time='',
        end_time='',
        count=1,
        dividend_type='none',
        period='1d',
        fill_data=False
    )
print(df)

{'ForwardFactor':             688318.SH
2026-04-16        1.0, 'Low':             688318.SH
2026-04-16     112.65, 'Amount':             688318.SH
2026-04-16   36365.11, 'Close':             688318.SH
2026-04-16     114.03, 'Open':             688318.SH
2026-04-16      113.6, 'Volume':             688318.SH
2026-04-16  3182209.0, 'High':             688318.SH
2026-04-16     115.59}


### 1.2 Equity Quote (股票实时报价)

In [4]:
'''
    获取市场快照数据,调用会触发客户端刷新数据，耗时过长请耐心等待
    总成交额为万位，其他无特殊说明均为个位
    曾用名：get_full_tick get_report_data
'''
market_snapshot = tq.get_market_snapshot(stock_code = '688260.SH', field_list=[])
print(market_snapshot)


{'ItemNum': '623', 'LastClose': '29.15', 'Open': '29.17', 'Max': '29.58', 'Min': '28.70', 'Now': '29.06', 'Volume': '12702', 'NowVol': '3', 'Amount': '3705.63', 'Inside': '6435', 'Outside': '6267', 'TickDiff': '0.00', 'InOutFlag': '1', 'Jjjz': '0.00', 'Buyp': ['29.06', '0.00', '0.00', '0.00', '0.00'], 'Buyv': ['2', '0', '0', '0', '0'], 'Sellp': ['29.07', '0.00', '0.00', '0.00', '0.00'], 'Sellv': ['10', '0', '0', '0', '0'], 'UpHome': '0', 'DownHome': '0', 'Before5MinNow': '29.09', 'Average': '29.17', 'XsFlag': '2', 'Zangsu': '-0.09', 'ZAFPre3': '6.21', 'ErrorId': '0'}


### 1.3 Historical Dividends (历史分红)

In [7]:
'''
    获取分红送配数据
'''
divid_factors = tq.get_divid_factors(
        stock_code='600028.SH',
        start_time='',
        end_time='')
print(divid_factors)

           Type  Bonus  AllotPrice  ShareBonus  Allotment
Date                                                     
2002-07-22    1   0.80         0.0         0.0        0.0
2002-09-23    1   0.20         0.0         0.0        0.0
2003-06-23    1   0.60         0.0         0.0        0.0
2003-09-22    1   0.30         0.0         0.0        0.0
2004-06-07    1   0.60         0.0         0.0        0.0
2004-09-21    1   0.40         0.0         0.0        0.0
2005-06-06    1   0.80         0.0         0.0        0.0
2005-09-21    1   0.40         0.0         0.0        0.0
2006-06-19    1   0.90         0.0         0.0        0.0
2006-09-14    1   0.40         0.0         0.0        0.0
2006-10-10    1   0.00         0.0         2.8        0.0
2007-06-18    1   1.10         0.0         0.0        0.0
2007-09-19    1   0.50         0.0         0.0        0.0
2008-06-16    1   1.15         0.0         0.0        0.0
2008-09-22    1   0.30         0.0         0.0        0.0
2009-06-15    

### 1.4 Equity Profile (股权资料)

In [16]:
'''
    获取基础财务数据 与专业财务数据有区别 不需要下载专业财务数据
    field_list可以筛选返回字段，默认返回全部字段 比如 field_list=['J_zgb','ActiveCapital'] 就是只取总股本和流通股本
    股本 资产 负债 利润 现金流量等数据均为万位
'''
fdc = tq.get_stock_info(stock_code='688318.SH', field_list=[])
print(fdc)


{'Name': '财富趋势', 'Unit': '100', 'VolBase': '158.93', 'MinPrice': '0.01', 'XsFlag': '2', 'Fz': ['570', '690', '780', '900', '900', '900', '900', '900'], 'DelayMin': '0', 'QHVolBaseRate': '0', 'HKVolBaseRate': '0', 'BelongHS300': '0', 'BelongHasKQZ': '0', 'BelongRZRQ': '1', 'BelongHSGT': '1', 'IsHKGP': '0', 'IsQH': '0', 'IsQQ': '0', 'IsSTGP': '0', 'IsQuitGP': '0', 'TodayDRFlag': '0', 'HSStockKind': '4', 'ActiveCapital': '25611.94', 'J_zgb': '25611.94', 'J_bg': '0.00', 'J_hg': '0.00', 'J_zzc': '405548.44', 'J_ldzc': '246807.22', 'J_gdzc': '883.50', 'J_wxzc': '1176.91', 'J_ldfz': '20462.46', 'J_cqfz': '73.23', 'J_zbgjj': '157998.02', 'J_jzc': '383971.84', 'J_yysy': '36809.85', 'J_yycb': '6316.93', 'J_yszk': '4343.33', 'J_yyly': '35730.76', 'J_tzsy': '8642.19', 'J_jyxjl': '17460.20', 'J_zxjl': '1795.57', 'J_ch': '37.17', 'J_lyze': '35316.45', 'J_shly': '31540.50', 'J_jly': '31540.30', 'J_wfply': '187325.09', 'J_jyl': '8.21', 'J_mgwfp': '7.31', 'J_mgsy': '1.23', 'J_mgsy2': '0.00', 'J_mggjj':

### 1.5 Equity Search (股票搜索)

In [ ]:
'''
    获取股票代码
    默认为全部A股
    0:自选股 1:持仓股
    5:所有A股 6:上证指数成份股 7:上证主板 8:深证主板 9:重点指数 
    10:所有板块指数 11:缺省行业板块 12:概念板块 13:风格板块 14:地区板块 15:缺省行业分类+概念板块 16:研究行业一级 17:研究行业二级 18:研究行业三级
    21:含H股 22:含可转债 23:沪深300 24:中证500 25:中证1000 26:国证2000 27:中证2000 28:中证A500
    30:REITs 31:ETF基金 32:可转债 33:LOF基金 34:所有可交易基金 35:所有沪深基金 36:T+0基金
    49:金融类企业 50:沪深A股 51:创业板 52:科创板 53:北交所
    101:国内期货 102:港股 103:美股    
'''

# 所有A股可以获得代码和名称
a_stock_list = tq.get_stock_list(market = "5", list_type=1)
print(a_stock_list)
print(len(a_stock_list))

# 102:港股 - 只能获得代码，不能获得名称。
h_stock_list = tq.get_stock_list(market = "102", list_type=0)
print(h_stock_list)
print(len(h_stock_list))


[{'Code': '000001.SZ', 'Name': '平安银行'}, {'Code': '000002.SZ', 'Name': '万 科Ａ'}, {'Code': '000004.SZ', 'Name': '*ST国华'}, {'Code': '000006.SZ', 'Name': '深振业Ａ'}, {'Code': '000007.SZ', 'Name': '全新好'}, {'Code': '000008.SZ', 'Name': '神州高铁'}, {'Code': '000009.SZ', 'Name': '中国宝安'}, {'Code': '000010.SZ', 'Name': '美丽生态'}, {'Code': '000011.SZ', 'Name': '深物业A'}, {'Code': '000012.SZ', 'Name': '南 玻Ａ'}, {'Code': '000014.SZ', 'Name': '沙河股份'}, {'Code': '000016.SZ', 'Name': '深康佳Ａ'}, {'Code': '000017.SZ', 'Name': '深中华A'}, {'Code': '000019.SZ', 'Name': '深粮控股'}, {'Code': '000020.SZ', 'Name': '深华发Ａ'}, {'Code': '000021.SZ', 'Name': '深科技'}, {'Code': '000025.SZ', 'Name': '特 力Ａ'}, {'Code': '000026.SZ', 'Name': '飞亚达'}, {'Code': '000027.SZ', 'Name': '深圳能源'}, {'Code': '000028.SZ', 'Name': '国药一致'}, {'Code': '000029.SZ', 'Name': '深深房Ａ'}, {'Code': '000030.SZ', 'Name': '富奥股份'}, {'Code': '000031.SZ', 'Name': '大悦城'}, {'Code': '000032.SZ', 'Name': '深桑达Ａ'}, {'Code': '000034.SZ', 'Name': '神州数码'}, {'Code': '000035.SZ', 'Name

## 2. 财务报表模型 (Financial Statement Models)
### 2.1 Key Metrics (关键指标)

Key Metrics is under `obb.equity.fundamental.metrics`.


In [8]:
from openbb_tdx.utils.tdx_key_metrics import map_tdx_to_openbb
from mysharelib.tools import normalize_symbol
symbol = '688318.SH'
symbol_b, symbol_f, market = normalize_symbol(symbol.strip())

mapped_data = map_tdx_to_openbb(symbol_f, auto_connect=False)
print(mapped_data)


{'symbol': '688318.SH', 'period_ending': '2026-03-31', 'fiscal_year': 2026, 'fiscal_period': 'Q1', 'currency': 'CNY', 'market_cap': 293080000000.0, 'pe_ratio': 92.92, 'forward_pe': 92.92, 'eps': 1.23, 'price_to_sales': 7961999.301817313, 'price_to_book': 7.63, 'book_value_per_share': 14.99, 'price_to_cash': 5504571.3034309475, 'cash_per_share': 2.0788358866997187, 'price_to_free_cash_flow': None, 'debt_to_equity': 0.053291564298048516, 'long_term_debt_to_equity': None, 'quick_ratio': 12.059647276036214, 'current_ratio': 12.061463773172923, 'gross_margin': 0.8283902270723733, 'profit_margin': 0.0821, 'operating_margin': 0.9706847487832742, 'return_on_assets': 0.07777196726487223, 'return_on_investment': None, 'return_on_equity': 0.08214222167958983, 'payout_ratio': None, 'dividend_yield': 0.0033, 'enterprise_value': 293079967219.44, 'ev_to_sales': 7961998.41127959, 'ev_to_ebitda': None, 'beta': 1.64, 'year_high': 180.86, 'year_low': 90.93, 'volume_avg': None, 'working_capital': 226344.7

### 2.2 Balance Sheet (资产负债表)

In [10]:
from openbb_tdx.utils.financial_statement_mapping import BalanceSheetMapper
year = 2024
mmdd = 331

'''
    按指定日期获取专业财务数据 
'''
balance_sheet_fields = BalanceSheetMapper.get_field_list()
balance_sheet_data = tq.get_financial_data_by_date(
        stock_list=[symbol_a],
        field_list=balance_sheet_fields,
        year=year,
        mmdd=mmdd)
print(balance_sheet_data)

mapped_all_balance = BalanceSheetMapper.map_from_get_financial_data_by_date(balance_sheet_data, year=year, mmdd=mmdd)
print(mapped_all_balance)


{'000001.SZ': {'FN10': '0.00', 'FN11': '0.00', 'FN12': '0.00', 'FN13': '0.00', 'FN14': '0.00', 'FN15': '0.00', 'FN16': '0.00', 'FN17': '0.00', 'FN18': '0.00', 'FN19': '0.00', 'FN20': '0.00', 'FN21': '0.00', 'FN22': '0.00', 'FN23': '0.00', 'FN24': '0.00', 'FN25': '0.00', 'FN26': '326000000.00', 'FN27': '9438000128.00', 'FN271': '485576015872.00', 'FN28': '0.00', 'FN29': '0.00', 'FN295': '0.00', 'FN296': '0.00', 'FN298': '2608999936.00', 'FN30': '0.00', 'FN31': '0.00', 'FN32': '0.00', 'FN33': '6438000128.00', 'FN34': '0.00', 'FN35': '7568000000.00', 'FN36': '0.00', 'FN37': '44308000768.00', 'FN38': '0.00', 'FN39': '0.00', 'FN40': '5729397768192.00', 'FN41': '0.00', 'FN42': '47220998144.00', 'FN43': '0.00', 'FN44': '0.00', 'FN45': '0.00', 'FN46': '14032000000.00', 'FN47': '11333999616.00', 'FN48': '0.00', 'FN49': '0.00', 'FN50': '0.00', 'FN51': '0.00', 'FN52': '0.00', 'FN53': '0.00', 'FN54': '0.00', 'FN55': '0.00', 'FN56': '758919987200.00', 'FN57': '0.00', 'FN58': '0.00', 'FN59': '108070

### 2.3 Income Statement (利润表)

In [6]:
from openbb_tdx.utils.financial_statement_mapping import IncomeStatementMapper
year = 0
mmdd = 0

'''
    按指定日期获取专业财务数据 
'''
income_statement_fields = IncomeStatementMapper.get_field_list()
income_statement_data = tq.get_financial_data_by_date(
        stock_list=['688318.SH'],
        field_list=income_statement_fields,
        year=year,
        mmdd=mmdd)
print(income_statement_data)

mapped_all_income = IncomeStatementMapper.map_from_get_financial_data_by_date(income_statement_data, year=year, mmdd=mmdd)
print(mapped_all_income)


{'688318.SH': {'FN1': '1.23', 'FN134': '315405120.00', 'FN135': '0.00', 'FN183': '-5.37', 'FN184': '3.77', 'FN197': '8.21', 'FN199': '85.68', 'FN2': '0.96', 'FN202': '82.84', 'FN206': '246968112.00', 'FN207': '328461632.00', 'FN208': '332772736.00', 'FN209': '90.40', 'FN230': '169820032.00', 'FN231': '148946944.00', 'FN232': '131189760.00', 'FN233': '149185952.00', 'FN238': '256119472.00'}}
{'688318.SH': {'query_year': 0, 'query_mmdd': 0, 'revenue': 169820032, 'operating_income': 148946944, 'net_income': 131189760, 'net_income_from_continuing_operations': 149185952, 'net_income_attributable_to_common_shareholders': 246968112, 'consolidated_net_income': 315405120, 'basic_earnings_per_share': 1.23, 'diluted_earnings_per_share': 0.96, 'depreciation_and_amortization': 0, 'ebit': 328461632, 'ebitda': 332772736, 'weighted_average_basic_shares_outstanding': 256119472, 'return_on_equity': 8.21, 'net_profit_margin': 0.7725222899498688, 'gross_margin': 82.84, 'revenue_growth': -5.37, 'net_income

### 2.4 Cash Flow Statement (现金流量表)

In [7]:
from openbb_tdx.utils.financial_statement_mapping import CashFlowStatementMapper
year = 0
mmdd = 0

'''
    按指定日期获取专业财务数据 
'''
cash_flow_statement_fields = CashFlowStatementMapper.get_field_list()
cash_flow_statement_data = tq.get_financial_data_by_date(
        stock_list=['688318.SH'],
        field_list=cash_flow_statement_fields,
        year=year,
        mmdd=mmdd)
print(cash_flow_statement_data)

mapped_all_cash_flow = CashFlowStatementMapper.map_from_get_financial_data_by_date(cash_flow_statement_data, year=year, mmdd=mmdd)
print(mapped_all_cash_flow)


{'688318.SH': {'FN100': '5375212.50', 'FN101': '415566016.00', 'FN102': '48138100.00', 'FN103': '77101496.00', 'FN104': '75069840.00', 'FN105': '40654532.00', 'FN106': '240963968.00', 'FN107': '174602064.00', 'FN108': '3234798848.00', 'FN109': '43159200.00', 'FN110': '0.00', 'FN111': '0.00', 'FN112': '0.00', 'FN113': '3277958144.00', 'FN114': '55450280.00', 'FN115': '3287000576.00', 'FN116': '0.00', 'FN117': '0.00', 'FN118': '3342450688.00', 'FN119': '-64492616.00', 'FN120': '0.00', 'FN121': '0.00', 'FN122': '0.00', 'FN123': '0.00', 'FN124': '0.00', 'FN125': '91402336.00', 'FN126': '751292.13', 'FN127': '92153624.00', 'FN128': '-92153624.00', 'FN129': '277718.09', 'FN130': '0.00', 'FN131': '18233536.00', 'FN132': '512196672.00', 'FN133': '530430208.00', 'FN134': '315405120.00', 'FN135': '0.00', 'FN136': '3774780.00', 'FN137': '309056.88', 'FN138': '227269.77', 'FN139': '0.00', 'FN140': '0.00', 'FN141': '-39102788.00', 'FN142': '-19172050.00', 'FN143': '-86421944.00', 'FN144': '1112747.

## 自选股

In [8]:
'''
    创建自定义板块
    block_code为板块简称 block_name为板块名称
'''
create_ptr = tq.create_sector(block_code='CSBK1', block_name='A股测试1')
print(create_ptr)


{"ErrorId":"0","Msg":"创建CSBK1板块成功","run_id":"0"}



In [10]:
'''
    添加自选股 到 通达信客户端的临时条件股列表
    block_code 为客户端已有的自定义板块简称，如果不存在则无效果，空则为添加到临时条件股
    block_code存在，传入空列表则表示清空该板块所有股票，否则为添加新股票
    自选股的block_code为ZXG
    shows 参数表示是否在客户端显示该自选股窗口
'''
zxg_result = tq.send_user_block(block_code='CSBK1', stocks=["600028.SH","600325.SH"], show=True)
# zxg_result = tq.send_user_block(block_code='CSBK', stocks=[])
# zxg_result = tq.send_user_block(block_code='', stocks=[])


In [3]:
'''
    手动断开连接 在策略退出前或错误处理中调用 比如取到数据为空 中途退出策略 调用close 这样才能配合TQ策略管理器关闭策略
    目前已经基本实现程序退出时自动断开连接，如国未能正常close会导致策略管理器中该策略运行状态始终处于运行状态，无法二次启动
    如果遇到这种情况，在策略管理器中删除该策略即可
'''
tq.close()


TQ数据连接已关闭
